# eigenfns — Maxwell eigenmodes of disordered LSU networks

Front-end for the GPU eigensolver. Workflow: **load structure → solve band window →
inspect spectrum & gap → render modes / montage**. Long solves belong in the CLI
(`scripts/run_modes.py`, checkpointed + resumable); this notebook is for interactive
exploration and uses the CLI's saved outputs when present.

Bands are **MPB-numbered** (bands 1–2 at Γ are the ω=0 modes). All physics and
validation records: `docs/REPORT_N1000.md`, `docs/plans/`.

In [ ]:
import os, sys
os.environ.setdefault("XLA_PYTHON_CLIENT_ALLOCATOR", "platform")
os.environ.setdefault("XLA_FLAGS", "--xla_gpu_enable_cublaslt=false --xla_gpu_autotune_level=0")
sys.path.insert(0, "..")
import numpy as np
import matplotlib.pyplot as plt
from eigenfns.structure import load_rods, rasterize_penlike
from eigenfns.operator import MaxwellOperator
from eigenfns.solver import lobpcg_blocks

STRUCTURE = ("/home/francisco/Documents/Create LSU Structures  - Claude/"
             "Example/N1000_lsu_example_ends.txt")
rods, N, L = load_rods(STRUCTURE)
print(f"N={N}, box L={L:.3f} µm, rods={len(rods)}")

## 1. Quick interactive solve (small grid)
A 64³ solve of the lowest ~100 bands takes ~2 min on the GPU — good for exploring.
For the production window (bands 398–607 at 128³, ~5.5 h) use the CLI.

In [ ]:
G = 64
eps = rasterize_penlike(rods, G, L)   # montage convention: binary, aspect 2.5, eps 8.57
op = MaxwellOperator(eps, L)
vals, vecs, stats = lobpcg_blocks(op, 96, m=32, guard=12, tol=1e-4, verbose=True)
nu = np.sqrt(vals) * 2.288 / (2 * np.pi)   # ν = ωa/2πc with a = srs cubic cell
plt.figure(figsize=(7, 3))
plt.plot(np.arange(1, len(nu) + 1) + 2, nu, ".", ms=3)   # +2: MPB numbering
plt.xlabel("band (MPB numbering)"); plt.ylabel("ν = ωa/2πc"); plt.tight_layout()

## 2. Production window results (from `scripts/run_modes.py`)
Loads the saved 128³ solve: full spectrum, the band gap, and window mode data.

In [ ]:
from pathlib import Path
RUN = Path("../results/prod_N1000_G128")
vals_all = np.load(RUN / "eigenvalues_all.npy")
nu_all = np.sqrt(vals_all) * 2.288 / (2 * np.pi)
bands = np.arange(1, len(nu_all) + 1) + 2
d = np.diff(vals_all)
gi = int(np.argmax(d[380:620]) + 380)
print(f"gap between MPB bands {gi+3} and {gi+4}: "
      f"ν {nu_all[gi]:.4f} → {nu_all[gi+1]:.4f} "
      f"(Δν/ν = {2*(nu_all[gi+1]-nu_all[gi])/(nu_all[gi+1]+nu_all[gi])*100:.2f}%)")
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].plot(bands, nu_all, ".", ms=2); ax[0].axvspan(gi+3, gi+4, color="orange", alpha=.3)
ax[0].set(xlabel="band", ylabel="ν", title="spectrum (128³)")
win = slice(380, 620)
ax[1].plot(bands[win], nu_all[win], ".", ms=3); ax[1].axvspan(gi+3, gi+4, color="orange", alpha=.3)
ax[1].set(xlabel="band", ylabel="ν", title="montage window 398–607")
plt.tight_layout()

## 3. Mode gallery: ε|E|² of selected bands
Below/inside/above the gap — extended vs localized character.

In [ ]:
ed = np.load(RUN / "window_energy_density.npy", mmap_mode="r")
meta_lo = 398   # first band stored in the window arrays
show = [420, 498, 500, 501, 503, 560]
fig, axes = plt.subplots(2, 3, figsize=(11, 7))
for ax, band in zip(axes.ravel(), show):
    f = np.asarray(ed[band - meta_lo])
    ax.imshow(f.max(axis=2).T, origin="lower", cmap="hot")
    ipr = float((f**2).sum() / f.sum()**2 * f.size)
    ax.set_title(f"band {band} (IPR·V {ipr:.0f})"); ax.axis("off")
plt.suptitle("ε|E|² max-projections"); plt.tight_layout()

## 4. 3-D tile render + montage
Single-tile render in the reference montage's style (pyvista, off-screen);
full 210-tile montage via `scripts/make_montage.py`.

In [ ]:
from eigenfns.render import render_tile
from IPython.display import Image as IPImage, display
eps128 = rasterize_penlike(rods, 128, L)
for band in (450, 500):
    out = f"/tmp/tile_band{band}.png"
    render_tile(eps128, np.asarray(ed[band - meta_lo]), out)
    display(IPImage(out, width=340))

## 5. Validation ledger
All pre-registered gates and their measured values (see docs/REPORT_N1000.md §3).

In [ ]:
import json
for name, rec in json.load(open("../results/gates/gate_results.json")).items():
    status = "SKIP" if "skip" in rec else ("PASS" if rec.get("pass") else "FAIL")
    print(f"[{status}] {name}")